**RSA algorithm demo**
----------------------

Modules import

In [15]:
import math
import random

**Step 1**: Function to create prime number of arbitrary length

Miller-Rabin primality test to determine whether a given number `n` is prime.with high pobability

*Error Rate <= 4^(-k)*

In [16]:
def is_probable_prime(n: int, k: int =10) -> bool:
    """
    Miller-Rabin primality test: determines whether a given number n is prime
    with high probability (error rate <= 4^(-k)).
    """
    if n == 2 or n == 3:
        return True
    if n <= 1 or n % 2 == 0:
        return False

    # write n-1 as 2^r * d, where d is odd
    r = 0
    d = n - 1
    while d % 2 == 0:
        r += 1
        d //= 2

    # run k rounds of the test
    for _ in range(k):
        a = random.randint(2, n-2)
        x = pow(a, d, n)
        if x == 1 or x == n-1:
            continue
        for _ in range(r-1):
            x = pow(x, 2, n)
            if x == n-1:
                break
        else:
            return False

    return True

Generating prime number of given length

In [17]:
def generate_prime(length: int) -> int:
    """
    Generate a random prime number of a given length in decimal digits.
    """
    lower_bound = 10**(length-1)
    upper_bound = 10**length - 1
    while True:
        n = random.randint(lower_bound, upper_bound)
        if is_probable_prime(n):
            return n

**Settings**

In [18]:
PRIME_LENGTH = 2 # keep it small for performance

**Step 1**: Choose two large prime numbers `P` and `Q`

Example:  
Let `P = 7` and `Q = 17`

In [19]:
P = generate_prime(PRIME_LENGTH)
Q = P

while Q == P:
    P = generate_prime(PRIME_LENGTH)

print(f'P = {P}')
print(f'Q = {Q}')

P = 41
Q = 53


**Step 2**: Calculate `N = P x Q`

Example:  
`N = 7 x 17 = 119`

In [20]:
N = P * Q
print(f'N = {N}')

N = 2173


**Step 3**: Select the public key (i.e. encryption key) `E` such that it is not a factor of `(P - 1) x (Q - 1)`

In other words, choose `E` such that `GCD(E, ϕ(N)) = 1`

where,
```
    ϕ(N) = ϕ(P) * ϕ(Q)  
    ϕ(N) = (P - 1) * (Q - 1)
```

Example:
- Let `P = 7` and `Q = 17`
- Let us find `(7 - 1) x (17 - 1) = 6 x 16 = 96`
- The factor of `96` are `2, 2, 2, 2, 2, 3`
- Thus, we have to choose `E` such that none of the factor of `E` is 2 and 3. As a few examples, we cannot choose `E` as 4 (because it has 2 as a factor), 15 (because it has 3 as a factor), 6 (because it has 2 and 3 both as factors).
- Let choose `E` as `5` (bracuse it don't have factors as 2 and 3)

In [21]:
phi_N = (P-1) * (Q-1)

while 1:
    E = random.randint(2, phi_N-1) # 1 < E < ϕ(N)
    if math.gcd(E, phi_N) == 1:
        break 

print(f'ϕ(N) = (P - 1) x (Q - 1) = {phi_N}')
print(f'E = {E}')

ϕ(N) = (P - 1) x (Q - 1) = 2080
E = 1723


**Step 4**: Select the private key (i.e. the decryption key) `D` such that following equation is true:
```
(D x E) mod (P-1) x (Q-1) = 1
```

In other words, choose `D` such that `(D x E) mod ϕ(n) = 1`

Example:
- Let use substitute the value of `E`, `P` and `Q` in the equation.
- We have `(D x 5) mod (7 - 1) x (17 - 1) = 1`
- That is, `(D x 5) mod (6) x (16) = 1`
- That is, `(D x 5) mod (96) = 1`
- After some calculations, let us take `D = 77`. Then the following is true:
```
(77 x 5) mod (96) = 385 mod 96 = 1
```

**NOTE**:
Brute forcing to find appropriate value of `D` is not efficient for small primes also.

**Calculate Modular Inverse**

To find `D`, calculate the modular inverse of `E mod ϕ(N)`  
In mathematical terms, you need to find `D` such that `D x E = 1 (mod ϕ(N))`

This can be achieved using the _Extended Euclidean Algorithm_. The algorithm will give you the value of `D` that satisfies the above congruence.

In [22]:
def extended_gcd(a, b):
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y

def modular_inverse(e, phi):
    gcd, x, y = extended_gcd(e, phi)
    if gcd != 1:
        raise ValueError("Modular inverse does not exist")
    return x % phi

D = modular_inverse(E, phi_N)

print(f'D = {D}')

D = 1107


**Step 5**: For encryption, calculate the cipher text CT from the plain text PT as follows:
```
CT = PT^E mod N
```

Example:
- Let us assume that we want to encrypt plain text `10`. Then we have: 
```CT = 10^5 mod 119 = 1_00_000 mod 119 = 40```

In [23]:
PT1 = 327
CT = (PT1**E) % N 

print(f'PT1 = {PT1}')
print(f'CT = {CT}')

PT1 = 327
CT = 1680


**Step 6**: Send cipher text `CT` to the receiver

**Step 7**: For decryption, calculate the plaint text PT from the cipher text CT as follows:
```
PT = CT^D mod N
```

Example:
- `PT = 40^77 mod 119 = 10` which was the originat plain text encrypted in _step 4_

In [24]:
PT2 = (CT**D) % N

print(f'PT2 = {PT2}')

PT2 = 327


**Conclusion**

In [25]:
print(
    f'Modulus (N): {N}',
    f'Private key (E): {E}',
    f'Public key (D): {D}',
    sep='\n',
)


Modulus (N): 2173
Private key (E): 1723
Public key (D): 1107


### **Encrypt | Decrypt**

Creating helper functions

In [26]:
char = str

def encrypt(ch: char, private_key: int, modulus: int, encoding: str = 'utf-8') -> int:
    cipher_int = int(bytes(ch, encoding=encoding).hex(), 16)
    return pow(cipher_int, private_key, modulus)

def decrypt(cipher: int, public_key: int, modulus: int, encoding: str = 'utf-8') -> char:
    cipher_hex = hex(pow(cipher, public_key, modulus))[2:]
    return bytes.fromhex(cipher_hex).decode(encoding=encoding)

__main__

In [27]:
letter = 'h'
encrypted_message = encrypt(letter, E, N)
decrypted_message = decrypt(encrypted_message, D, N)

print(decrypted_message)

h
